# Clase 03 — Actividad personal (reestructurada + Optuna)

**Objetivo del profe:**
1. Comparar 2 hiperparámetros.
2. Lo mismo, pero con el test en marzo.
3. ¿El mejor de marzo es el mejor de mayo?

**Extra:** optimización de hiperparámetros del árbol con **Optuna**.

**Capas:**
- **Capa 1 — Base (tuya / cátedra):** imports, constantes, datos, semillas, `ganancia`. Intacto.
- **Capa 2 — Infraestructura:** funciones que envuelven tu loop de siempre.
- **Capa 3 — Análisis y decisión:** comparación de configs, marzo vs mayo y Optuna.

> Todos los gráficos son **histogramas de distribución**. Tu `ganancia` y el
> filtrado por `foto_mes` no se tocan.


## Capa 1 — Base (sin cambios)

In [ ]:
import sys

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.tree import DecisionTreeClassifier
import seaborn as sns

sys.path.insert(0, "..")
from config.semillas import SEMILLA_PRIMARIA

DATA_PATH = "../datasets/processed/competencia_01.csv"

con = duckdb.connect()
con.execute(f"""
    create or replace view labeled as
    select * exclude (numero_de_cliente, foto_mes)
    from read_csv_auto('{DATA_PATH}')
    where clase_ternaria is not null
""")

In [ ]:
ganancia_acierto = 1_072_500
costo_estimulo = 27_500

In [ ]:
data = pd.read_csv(DATA_PATH)

In [ ]:
# Semillas (las tuyas)
semillas = [590772946, 673765742, 293750017, 943403192, 979156448, 584538485, 452449729, 558697969,
            57360287, 130446541, 605182907, 950224428, 218289405, 939948071, 100863175, 414478288,
            160303261, 273512111, 865860863, 213714567, 808915688, 722029742, 925723517, 945671630,
            664520372, 54401700, 342474660, 846731114, 422760380, 578609822, 767778681, 83891579,
            527049832, 648818020, 55070016, 555906589, 258579277, 746504435, 633736736, 177637542,
            188273233, 321487413, 786466940, 648593534, 692127408, 894074607, 198169439, 586783997,
            999893340, 749684372, 42088082, 113747440, 372302911, 134610206, 888433608, 550860016,
            15725572, 326369614, 317756119, 919179051, 627565314, 43229677, 374383345, 783812080,
            906904853, 392284205, 162535622, 769809838, 712115623, 411443258, 634859221, 196005885,
            444086051, 797383292, 840184399, 790833327, 34990967, 888278840, 717241854, 564003472,
            526352615, 647777869, 928913153, 804729752, 432939581, 351356858, 308382688, 193177234,
            686379225, 787197794, 123401333, 501882704, 787535958, 799164038, 491576899, 285782967,
            638402799, 695124778, 909121796, 893364907]

In [ ]:
# Función de ganancia (de la cátedra) — INTACTA
def ganancia(model, X, y, prop=1, threshold=0.025):

  class_index = np.where(model.classes_ == "BAJA+2")[0][0]
  y_hat = model.predict_proba(X)

  @np.vectorize
  def ganancia_row(predicted, actual, threshold=0.025):
    return  (predicted >= threshold) * (ganancia_acierto if actual == "BAJA+2" else -costo_estimulo)

  return ganancia_row(y_hat[:,class_index], y).sum() / prop

## Capa 2 — Infraestructura de experimentación

`evaluar_config` es exactamente tu loop, encapsulado y llamando a **tu** `ganancia`.


In [ ]:
def datos_de_mes(data, mes):
    """Filtra al mes indicado y devuelve (X, y). Tu patrón de filtrado por foto_mes."""
    d = data[data['foto_mes'] == mes]
    y = d['clase_ternaria']
    X = d.drop(columns=['clase_ternaria', 'foto_mes'], errors='ignore')
    return X, y


def evaluar_config(X, y, tree_params, n_splits=10, test_size=0.3, seed=None):
    """Tu loop de validación repetida. Devuelve ganancias test/train (una por split)."""
    sss = StratifiedShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=seed)
    gan_test, gan_train = [], []
    for train_index, test_index in sss.split(X, y):
        model = DecisionTreeClassifier(**tree_params, random_state=seed)  # semilla también en el árbol
        model.fit(X.iloc[train_index], y.iloc[train_index])
        gan_test.append(ganancia(model, X.iloc[test_index],  y.iloc[test_index],  prop=test_size))
        gan_train.append(ganancia(model, X.iloc[train_index], y.iloc[train_index], prop=1 - test_size))
    return gan_test, gan_train


def correr_experimento(X, y, configs, seed):
    """Corre todas las configs sobre los mismos splits. Devuelve df_largo, resumen y ganancias de test por config."""
    filas, por_split_test = [], {}
    for nombre, params in configs.items():
        gt, gtr = evaluar_config(X, y, params, seed=seed)
        por_split_test[nombre] = np.array(gt)
        for g in gt:  filas.append({'Config': nombre, 'Conjunto': 'Test',  'Ganancia': g})
        for g in gtr: filas.append({'Config': nombre, 'Conjunto': 'Train', 'Ganancia': g})
    df_largo = pd.DataFrame(filas)

    test = df_largo[df_largo['Conjunto'] == 'Test']
    resumen = (test.groupby('Config')['Ganancia']
                   .agg(media='mean', std='std', mediana='median')
                   .sort_values('media', ascending=False))
    resumen['score'] = resumen['media'] - resumen['std']  # premia media, castiga inestabilidad
    return df_largo, resumen, por_split_test

## Capa 3.1 — Comparar 2 hiperparámetros (puntos 1 y 2)

Definí las configs en un diccionario. En el ejemplo se mueve `min_samples_split`
(80 vs 40); cambiá el hiperparámetro que quieras. **Vos elegís qué comparar.**


In [ ]:
MES = 202103  # test dentro de marzo (validación interna con StratifiedShuffleSplit)

X, y = datos_de_mes(data, MES)

configs = {
    'mss_80': {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 80},
    'mss_40': {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 40},
}

df_largo, resumen, por_split = correr_experimento(X, y, configs, seed=semillas[0])
resumen.round(0)

### Distribución de ganancia en Test por configuración (histograma)

Un histograma por config superpuesto por color: ves dónde se concentra la
ganancia de cada una y cuánto se solapan. `common_norm=False` normaliza cada
grupo por separado para que sean comparables.


In [ ]:
test_only = df_largo[df_largo['Conjunto'] == 'Test']

g = sns.displot(
    test_only, x='Ganancia', hue='Config',
    kind='hist', kde=True, palette='viridis',
    aspect=1.6, height=4.5, common_norm=False, alpha=0.5
)
g.set_axis_labels('Ganancia', 'Frecuencia')
g.figure.suptitle(f'Distribución de ganancia en Test por configuración (mes {MES})')
g.figure.subplots_adjust(top=0.9)
plt.show()

### Distribución Train vs Test (brecha de overfitting)

El mismo estilo histograma, ahora separando Train y Test por config. Si la
distribución de Train está muy corrida a la derecha respecto de Test, hay
overfitting.


In [ ]:
g = sns.displot(
    df_largo, x='Ganancia', hue='Conjunto', col='Config',
    kind='hist', kde=True, palette='viridis',
    aspect=1.1, height=4.5, common_norm=False, alpha=0.5
)
g.set_axis_labels('Ganancia', 'Frecuencia')
g.figure.suptitle('Train vs Test por configuración')
g.figure.subplots_adjust(top=0.85)
plt.show()

### El "valor" para decidir: comparación pareada

Como las configs se evaluaron sobre **los mismos 10 splits**, se restan split a
split. Ganar en 9/10 splits es señal más fuerte que un promedio más alto.


In [ ]:
nombres = list(configs.keys())
a, b = nombres[0], nombres[1]

dif = por_split[a] - por_split[b]
print(f'{a} vs {b}')
print(f'  {a} gana en {(dif > 0).sum()} de {len(dif)} splits')
print(f'  diferencia media   : {dif.mean():,.0f}')
print(f'  diferencia mediana : {np.median(dif):,.0f}')
print('\nResumen ordenado por score (media - std):')
print(resumen.round(0))

## Capa 3.2 — ¿El mejor de marzo es el mejor de mayo?

Se corre el mismo experimento (mismas configs, misma semilla) en los dos meses y
se comparan los **rankings**. La pregunta no es solo *cuál gana*, sino **si el
orden se mantiene** entre meses:

- Ranking estable → elección robusta al período.
- Ganador de marzo cae en mayo → sobreajuste al mes de validación. En datos con
  drift mensual eso es esperable, y **reportarlo es la respuesta que busca el profe.**


In [ ]:
def experimento_por_mes(data, mes, configs, seed):
    X, y = datos_de_mes(data, mes)
    df_largo, resumen, _ = correr_experimento(X, y, configs, seed)
    df_largo = df_largo.copy()
    df_largo['Mes'] = str(mes)
    return df_largo, resumen


dfl_mar, res_marzo = experimento_por_mes(data, 202103, configs, semillas[0])
dfl_may, res_mayo  = experimento_por_mes(data, 202105, configs, semillas[0])
df_meses = pd.concat([dfl_mar, dfl_may], ignore_index=True)

print('=== MARZO (202103) ===');  print(res_marzo.round(0))
print('\n=== MAYO (202105) ===');  print(res_mayo.round(0))

In [ ]:
# Comparación de rankings lado a lado
comp = pd.DataFrame({
    'media_marzo': res_marzo['media'],
    'rank_marzo':  res_marzo['media'].rank(ascending=False).astype(int),
    'media_mayo':  res_mayo['media'],
    'rank_mayo':   res_mayo['media'].rank(ascending=False).astype(int),
}).sort_values('rank_marzo')

ganador_marzo = comp.index[comp['rank_marzo'] == 1][0]
ganador_mayo  = comp.index[comp['rank_mayo']  == 1][0]

print(comp.round(0))
print(f'\nMejor en marzo: {ganador_marzo}')
print(f'Mejor en mayo : {ganador_mayo}')
print('=> El mejor de marzo', 'SÍ' if ganador_marzo == ganador_mayo else 'NO', 'es el mejor de mayo.')

### Distribución de ganancia por mes (histograma)

Un panel por mes, cada config en un color. Si el color que domina (más a la
derecha) cambia de un panel al otro, el ganador cambió entre meses.


In [ ]:
test_meses = df_meses[df_meses['Conjunto'] == 'Test']

g = sns.displot(
    test_meses, x='Ganancia', hue='Config', col='Mes',
    kind='hist', kde=True, palette='viridis',
    aspect=1.1, height=4.5, common_norm=False, alpha=0.5
)
g.set_axis_labels('Ganancia', 'Frecuencia')
g.figure.suptitle('Distribución de ganancia en Test: Marzo vs Mayo')
g.figure.subplots_adjust(top=0.85)
plt.show()

## Capa 3.3 — Optimización de hiperparámetros con Optuna

Optuna busca la mejor combinación con optimización bayesiana (TPE), mucho más
eficiente que probar a mano. La pieza clave es el **`objective`**: propone
hiperparámetros, corre **tu** `evaluar_config` con ellos y devuelve la **media de
ganancia en test** (lo que se maximiza).

Notas de método:
- En la búsqueda se usan **5 splits** (más rápido); el ganador se reevalúa con
  10 al final.
- Acá se optimiza con **una semilla** para que la clase corra rápido. Para el
  proyecto, lo correcto es promediar el `objective` sobre tus **5 semillas prime**
  (queda indicado abajo) para no sobreajustar a una sola partición.
- `class_weight` entra como hiperparámetro porque tu problema (BAJA+2 es raro) es
  desbalanceado y puede ayudar bastante.

> Requiere `optuna` instalado: `pip install optuna` en tu `.venv`.


In [ ]:
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)  # menos ruido en la salida

MES_OPT = 202103
X_opt, y_opt = datos_de_mes(data, MES_OPT)


def objective(trial):
    params = {
        'criterion':             trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'max_depth':             trial.suggest_int('max_depth', 3, 20),
        'min_samples_split':     trial.suggest_int('min_samples_split', 2, 400),
        'min_samples_leaf':      trial.suggest_int('min_samples_leaf', 1, 200),
        'max_leaf_nodes':        trial.suggest_int('max_leaf_nodes', 8, 256),
        'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 0.01),
        'class_weight':          trial.suggest_categorical('class_weight', [None, 'balanced']),
    }
    # búsqueda con 5 splits para ir más rápido
    gan_test, _ = evaluar_config(X_opt, y_opt, params, n_splits=5, seed=semillas[0])
    return float(np.mean(gan_test))


study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=semillas[0]))
study.optimize(objective, n_trials=20, show_progress_bar=True)

print('Mejor ganancia media (test, 5 splits):', f'{study.best_value:,.0f}')
print('Mejores hiperparámetros:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

### Versión robusta del objective (para el proyecto)

Para no sobreajustar a una sola semilla, se promedia sobre varias. Es más lento,
así que se deja como referencia; descomentá para usarla.


In [ ]:
# def objective_multiseed(trial):
#     params = {
#         'criterion':             trial.suggest_categorical('criterion', ['gini', 'entropy']),
#         'max_depth':             trial.suggest_int('max_depth', 3, 20),
#         'min_samples_split':     trial.suggest_int('min_samples_split', 2, 400),
#         'min_samples_leaf':      trial.suggest_int('min_samples_leaf', 1, 200),
#         'max_leaf_nodes':        trial.suggest_int('max_leaf_nodes', 8, 256),
#         'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 0.01),
#         'class_weight':          trial.suggest_categorical('class_weight', [None, 'balanced']),
#     }
#     medias = []
#     for s in semillas[:5]:                      # tus 5 semillas prime
#         gt, _ = evaluar_config(X_opt, y_opt, params, n_splits=5, seed=s)
#         medias.append(np.mean(gt))
#     return float(np.mean(medias))
#
# study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=semillas[0]))
# study.optimize(objective_multiseed, n_trials=50, show_progress_bar=True)

### Comparar el mejor de Optuna contra tu baseline (histograma)

Reevaluamos el mejor con **10 splits** y lo comparamos con tu config base, en el
mismo estilo histograma.


In [ ]:
best_params = study.best_params

gan_best, _ = evaluar_config(X_opt, y_opt, best_params,       n_splits=10, seed=semillas[0])
gan_base, _ = evaluar_config(X_opt, y_opt, configs['mss_80'], n_splits=10, seed=semillas[0])

df_cmp = pd.DataFrame({
    'Ganancia': gan_best + gan_base,
    'Modelo': ['Optuna best'] * len(gan_best) + ['Baseline mss_80'] * len(gan_base)
})

print('Media Optuna best :', f'{np.mean(gan_best):,.0f}')
print('Media baseline    :', f'{np.mean(gan_base):,.0f}')
print('Mejora            :', f'{np.mean(gan_best) - np.mean(gan_base):,.0f}')

g = sns.displot(
    df_cmp, x='Ganancia', hue='Modelo',
    kind='hist', kde=True, palette='viridis',
    aspect=1.6, height=4.5, common_norm=False, alpha=0.5
)
g.set_axis_labels('Ganancia', 'Frecuencia')
g.figure.suptitle('Optuna best vs Baseline (Test, 10 splits)')
g.figure.subplots_adjust(top=0.9)
plt.show()

### ¿El óptimo de marzo sobrevive en mayo?

El mismo test de robustez de la Capa 3.2, ahora aplicado al ganador de Optuna:
lo evaluamos en mayo y lo comparamos con marzo (histograma). Si la distribución
en mayo se desploma, el óptimo estaba sobreajustado a marzo.


In [ ]:
X_may, y_may = datos_de_mes(data, 202105)
gan_best_may, _ = evaluar_config(X_may, y_may, best_params, n_splits=10, seed=semillas[0])

df_best_meses = pd.DataFrame({
    'Ganancia': gan_best + gan_best_may,
    'Mes': ['202103'] * len(gan_best) + ['202105'] * len(gan_best_may)
})

print('Optuna best en marzo:', f'{np.mean(gan_best):,.0f}')
print('Optuna best en mayo :', f'{np.mean(gan_best_may):,.0f}')

g = sns.displot(
    df_best_meses, x='Ganancia', hue='Mes',
    kind='hist', kde=True, palette='viridis',
    aspect=1.6, height=4.5, common_norm=False, alpha=0.5
)
g.set_axis_labels('Ganancia', 'Frecuencia')
g.figure.suptitle('Óptimo de marzo evaluado en marzo vs mayo')
g.figure.subplots_adjust(top=0.9)
plt.show()

## Cómo leer todo esto para decidir

1. **No mires solo la media.** Config con media apenas menor pero más estable
   (menor `std`) suele ser mejor apuesta → columna `score`.
2. **La comparación pareada manda.** Ganar en 9/10 splits > promedio más alto.
3. **La brecha train–test** te avisa de overfitting aunque el test se vea bien.
4. **La prueba final es la estabilidad entre meses.** Vale tanto para las configs
   a mano como para el óptimo de Optuna: si no sobrevive de marzo a mayo, la
   conclusión del ejercicio es esa.

**Optuna, con criterio:**
- Es una herramienta de búsqueda, no un oráculo. El `objective` optimiza lo que
  vos le pongas; si optimiza sobre un solo mes/semilla, hereda ese sesgo.
- Herramientas útiles del propio Optuna para inspeccionar (no son histogramas, por
  eso quedan como mención): `optuna.visualization.plot_optimization_history(study)`
  y `optuna.visualization.plot_param_importances(study)` para ver qué
  hiperparámetro pesó más.
